# 02 · Train and evaluate the U-Net (V5)

Semantic segmentation of 6 LULC classes from Sentinel-1 VV/VH only.

* **Model:** U-Net, 4 levels, 16 base channels, 1.94 M parameters
* **Loss:** class-weighted cross-entropy (ignore index 255)
* **Sampler:** patches containing ≥ 100 bare or water pixels drawn twice as often
* **Optimiser:** AdamW, lr 1e-3, weight decay 1e-4, batch 4, 30 epochs
* **Model selection:** best validation mIoU
* **Test set:** spatially held-out locations, evaluated once

Set `RETRAIN = False` to skip training and rebuild every figure from the saved
checkpoint and `training_history_v5.json`.

In [ ]:
import json, sys, time, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_DIR))

from src.model import UNet
from src.classes import CLASS_NAMES, CLASS_COLORS, CLASS_IDS, IGNORE_INDEX
from src.visualize import CMAP, NORM, LEGEND

# Dataset used for V5. Switch to data/patches/{train,val,test}.npz after running notebook 01.
TRAIN_PATH = PROJECT_DIR / "patches_v5" / "train_v5.npz"
VAL_PATH   = PROJECT_DIR / "patches_v4" / "val_v4.npz"
TEST_PATH  = PROJECT_DIR / "patches_v4" / "test_v4.npz"

OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT = OUTPUT_DIR / "best_unet_v5.pt"
HISTORY    = OUTPUT_DIR / "training_history_v5.json"
FIG_DIR    = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

RETRAIN = not (CHECKPOINT.exists() and HISTORY.exists())

SEED, BATCH_SIZE, EPOCHS, LR, WEIGHT_DECAY = 42, 4, 30, 1e-3, 1e-4
NUM_WORKERS = 4
CLASS_WEIGHTS = [0.48, 0.48, 0.48, 0.48, 2.398, 1.684]
NAMES = [CLASS_NAMES[c] for c in CLASS_IDS]

torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")
print("PyTorch:", torch.__version__)
print("RETRAIN:", RETRAIN)
for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    print(f"  {p.relative_to(PROJECT_DIR)}: {'found' if p.exists() else 'MISSING'}")

## 1. Data

In [ ]:
def load(path):
    with np.load(path, allow_pickle=False) as d:
        X = d["X"].astype(np.float32)
        y = d["y"].astype(np.int64)
        meta = d["metadata"] if "metadata" in d else None
        stats = (d["class_medians"].astype(np.float32), d["class_iqr"].astype(np.float32))
    y = np.where(y == IGNORE_INDEX, IGNORE_INDEX, y - 1)   # class IDs 1-6 -> 0-5
    return X, y, meta, stats

X_train, y_train, meta_train, stats = load(TRAIN_PATH)
X_val, y_val, meta_val, val_stats = load(VAL_PATH)
X_test, y_test, meta_test, _ = load(TEST_PATH)
assert np.allclose(stats[0], val_stats[0]) and np.allclose(stats[1], val_stats[1]), "Normalisation mismatch"

print(f"Train {X_train.shape}  Val {X_val.shape}  Test {X_test.shape}")
print("Normalisation  median:", stats[0], " IQR:", stats[1])

counts = pd.DataFrame({
    split: [(y == k).sum() for k in range(6)]
    for split, y in [("train", y_train), ("val", y_val), ("test", y_test)]
}, index=NAMES)
display((counts / counts.sum() * 100).round(2).add_suffix(" (%)"))

ax = (counts / counts.sum() * 100).plot.bar(figsize=(10, 3.8), color=["#7f8c8d", "#2980b9", "#c0392b"])
ax.set_ylabel("Labelled pixels (%)"); ax.set_title("Class distribution")
plt.xticks(rotation=20, ha="right"); plt.tight_layout()
plt.savefig(FIG_DIR / "class_distribution.png", dpi=130); plt.show()

In [ ]:
rare = ((y_train == 4).sum(axis=(1, 2)) >= 100) | ((y_train == 5).sum(axis=(1, 2)) >= 100)
weights = np.where(rare, 2.0, 1.0)
g = torch.Generator().manual_seed(SEED)
sampler = WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), len(weights),
                                replacement=True, generator=g)

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")
train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
                          sampler=sampler, **loader_kw)
val_loader = DataLoader(TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val)),
                        shuffle=False, **loader_kw)
test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test)),
                         shuffle=False, **loader_kw)
print(f"Oversampled patches: {int(rare.sum())} / {len(rare)} | batches per epoch: {len(train_loader)}")

## 2. Model, loss, optimiser

In [ ]:
model = UNet(in_channels=2, num_classes=6, base_channels=16).to(device)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(CLASS_WEIGHTS, device=device),
                                ignore_index=IGNORE_INDEX)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    cm = torch.zeros(36, dtype=torch.int64, device=device)
    total, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        logits = model(xb)
        total += criterion(logits, yb).item(); n += 1
        valid = yb != IGNORE_INDEX
        cm += torch.bincount(yb[valid] * 6 + logits.argmax(1)[valid], minlength=36)
    cm = cm.reshape(6, 6).cpu().numpy()
    return total / n, metrics_from_cm(cm), cm

def metrics_from_cm(cm):
    tp = np.diag(cm).astype(float)
    actual, pred = cm.sum(1).astype(float), cm.sum(0).astype(float)
    union = actual + pred - tp
    iou = np.divide(tp, union, out=np.zeros(6), where=union > 0)
    return {
        "accuracy": tp.sum() / actual.sum(),
        "miou": iou.mean(),
        "iou": iou,
        "precision": np.divide(tp, pred, out=np.zeros(6), where=pred > 0),
        "recall": np.divide(tp, actual, out=np.zeros(6), where=actual > 0),
    }

## 3. Training

In [ ]:
if RETRAIN:
    history = {k: [] for k in ["train_loss", "val_loss", "val_accuracy", "val_miou", "epoch_seconds"]}
    best = -1.0
    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        model.train()
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running += loss.item()

        val_loss, m, _ = evaluate(model, val_loader)
        history["train_loss"].append(running / len(train_loader))
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(m["accuracy"])
        history["val_miou"].append(m["miou"])
        history["epoch_seconds"].append(time.time() - t0)
        HISTORY.write_text(json.dumps(history, indent=2))

        improved = m["miou"] > best
        if improved:
            best = m["miou"]
            torch.save({
                "epoch": epoch, "model_state_dict": model.state_dict(), "best_val_miou": best,
                "architecture": "UNet", "in_channels": 2, "num_classes": 6, "base_channels": 16,
                "class_medians": stats[0].tolist(), "class_iqr": stats[1].tolist(),
                "class_names": NAMES, "seed": SEED,
            }, CHECKPOINT)
        print(f"Epoch {epoch:02d}/{EPOCHS} | train {history['train_loss'][-1]:.4f} | "
              f"val {val_loss:.4f} | acc {m['accuracy']:.4f} | mIoU {m['miou']:.4f} | "
              f"{history['epoch_seconds'][-1] / 60:.1f} min{'  <- best' if improved else ''}")
else:
    print("Skipping training; using", CHECKPOINT.relative_to(PROJECT_DIR))

history = json.loads(HISTORY.read_text())
ckpt = torch.load(CHECKPOINT, map_location=device, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Best epoch: {ckpt['epoch']} | best validation mIoU: {ckpt['best_val_miou']:.4f}")

log = pd.DataFrame(history)
log.index = np.arange(1, len(log) + 1); log.index.name = "epoch"
display(log.drop(columns="epoch_seconds").round(4))

## 4. Training curves

In [ ]:
epochs = np.arange(1, len(history["train_loss"]) + 1)
best_epoch = ckpt["epoch"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
axes[0].plot(epochs, history["train_loss"], "o-", label="train")
axes[0].plot(epochs, history["val_loss"], "o-", label="validation")
axes[0].set_title("Weighted cross-entropy loss"); axes[0].legend()
axes[1].plot(epochs, history["val_accuracy"], "o-", color="#27ae60")
axes[1].set_title("Validation pixel accuracy")
axes[2].plot(epochs, history["val_miou"], "o-", color="#8e44ad")
axes[2].set_title("Validation mean IoU")
for ax in axes:
    ax.axvline(best_epoch, color="grey", ls="--", lw=1)
    ax.set_xlabel("Epoch"); ax.grid(alpha=0.3)
axes[2].annotate(f"best epoch {best_epoch}", (best_epoch, max(history["val_miou"])),
                 xytext=(8, -14), textcoords="offset points", fontsize=9)
plt.tight_layout(); plt.savefig(FIG_DIR / "training_curves.png", dpi=130); plt.show()

## 5. Validation results (best checkpoint)

In [ ]:
def plot_confusion(cm, title, path):
    norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(6), NAMES, rotation=35, ha="right"); ax.set_yticks(range(6), NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Reference (WorldCover)")
    for (r, c), v in np.ndenumerate(norm):
        ax.text(c, r, f"{v:.2f}", ha="center", va="center", color="white" if v > 0.5 else "black", fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046, label="Row-normalised (recall)")
    ax.set_title(title); plt.tight_layout(); plt.savefig(path, dpi=130); plt.show()

def metric_table(m):
    return pd.DataFrame({"IoU": m["iou"], "Precision": m["precision"], "Recall": m["recall"]}, index=NAMES).round(4)

val_loss, val_m, val_cm = evaluate(model, val_loader)
print(f"Validation  loss {val_loss:.4f} | accuracy {val_m['accuracy']:.4f} | mIoU {val_m['miou']:.4f}")
display(metric_table(val_m))
plot_confusion(val_cm, "Validation confusion matrix", FIG_DIR / "confusion_validation.png")

## 6. Held-out test results

In [ ]:
test_loss, test_m, test_cm = evaluate(model, test_loader)
print(f"Test  loss {test_loss:.4f} | accuracy {test_m['accuracy']:.4f} | mIoU {test_m['miou']:.4f}")
test_table = metric_table(test_m)
display(test_table)

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(NAMES, test_m["iou"], color=[CLASS_COLORS[c] for c in CLASS_IDS])
ax.axhline(test_m["miou"], color="black", ls="--", lw=1, label=f"mIoU {test_m['miou']:.3f}")
for i, v in enumerate(test_m["iou"]):
    ax.text(i, v + 0.01, f"{v:.2f}", ha="center")
ax.set_ylim(0, 1); ax.set_ylabel("IoU"); ax.set_title("Test IoU per class"); ax.legend()
plt.xticks(rotation=20, ha="right"); plt.tight_layout()
plt.savefig(FIG_DIR / "test_iou_per_class.png", dpi=130); plt.show()

plot_confusion(test_cm, "Test confusion matrix", FIG_DIR / "confusion_test.png")

json.dump({"loss": test_loss, "accuracy": test_m["accuracy"], "miou": test_m["miou"],
           "per_class": test_table.to_dict(orient="index"), "confusion": test_cm.tolist()},
          open(OUTPUT_DIR / "test_metrics_v5.json", "w"), indent=2)

## 7. Test performance by month

In [ ]:
rows = []
months = sorted(np.unique(meta_test["month"]).tolist())
for month in months:
    idx = np.where(meta_test["month"] == month)[0]
    loader = DataLoader(TensorDataset(torch.from_numpy(X_test[idx]), torch.from_numpy(y_test[idx])),
                        batch_size=BATCH_SIZE)
    _, m, _ = evaluate(model, loader)
    rows.append({"month": month, "accuracy": m["accuracy"], "mIoU": m["miou"],
                 **{n: v for n, v in zip(NAMES, m["iou"])}})
monthly = pd.DataFrame(rows).set_index("month")
display(monthly.round(4))

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(monthly.index, monthly["mIoU"], "o-", color="black", lw=2.5, label="mIoU")
for c in CLASS_IDS:
    if CLASS_NAMES[c] != "Bare/sparse vegetation":
        ax.plot(monthly.index, monthly[CLASS_NAMES[c]], ".-", color=CLASS_COLORS[c], label=CLASS_NAMES[c])
ax.set_ylabel("IoU"); ax.set_ylim(0, 1); ax.grid(alpha=0.3); ax.legend(ncol=3, fontsize=9)
ax.set_title("Test IoU by month (seasonality of the radar signal)")
plt.xticks(rotation=45); plt.tight_layout()
plt.savefig(FIG_DIR / "test_iou_by_month.png", dpi=130); plt.show()

## 8. Example predictions (test set)

In [ ]:
@torch.no_grad()
def predict(x):
    model.eval()
    return model(torch.from_numpy(x[None]).to(device)).argmax(1)[0].cpu().numpy() + 1

picks = np.linspace(0, len(X_test) - 1, 4).astype(int)
fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for j, i in enumerate(picks):
    axes[0, j].imshow(X_test[i, 0], cmap="gray", vmin=-3, vmax=3)
    axes[0, j].set_title(f"VV  ({meta_test['month'][i]})")
    axes[1, j].imshow(np.ma.masked_equal(np.where(y_test[i] == IGNORE_INDEX, IGNORE_INDEX, y_test[i] + 1), IGNORE_INDEX),
                      cmap=CMAP, norm=NORM, interpolation="nearest")
    axes[1, j].set_title("WorldCover reference")
    axes[2, j].imshow(predict(X_test[i]), cmap=CMAP, norm=NORM, interpolation="nearest")
    axes[2, j].set_title("U-Net prediction")
for ax in axes.flat: ax.set_axis_off()
fig.legend(handles=LEGEND, loc="lower center", ncol=6, frameon=False)
plt.tight_layout(rect=(0, 0.04, 1, 1)); plt.savefig(FIG_DIR / "test_examples.png", dpi=130); plt.show()

## 9. Export weights for the prototype

In [ ]:
weights_dir = PROJECT_DIR / "weights"
weights_dir.mkdir(exist_ok=True)
shutil.copy2(CHECKPOINT, weights_dir / "best_unet_v5.pt")
print("Copied to", (weights_dir / "best_unet_v5.pt").relative_to(PROJECT_DIR),
      f"({(weights_dir / 'best_unet_v5.pt').stat().st_size / 1e6:.1f} MB)")

## Notes

* **Bare/sparse vegetation** has almost no reference pixels around Stuttgart and is not learned (IoU ≈ 0).
* **Grassland** is mostly confused with tree cover and cropland; in C-band radar these share similar backscatter.
* WorldCover 2021 is a static reference, so per-month scores measure agreement with one annual label,
  not true monthly ground truth.